## $z(t) = - t - Ae^{-2\kappa t} + B$

In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from scipy.special import gamma, loggamma

np.set_printoptions(precision=2, suppress=True)

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# discretization
N = 1000        # time samples
M = 10        # p, omega modes (natural numbers)

EPOCHS = 100    
BATCH_SIZE = 1  
LR = 1e-3

In [3]:
p_grid     = np.arange(1, M + 1)      # p = 1,2,...,10
omega_grid = np.arange(1, M + 1)      # ω = 1,2,...,10

In [4]:
def sample_parameters():
    A = np.random.uniform(0.1, 1.0)
    B = np.random.uniform(0.1, 1.0)
    kappa = np.random.uniform(0.3, 2)
    return A, B, kappa

In [5]:
def mirror_trajectory(t, A, B, kappa):
    return -t - A * np.exp(-2 * kappa * t) + B

In [6]:
def compute_alpha_beta(A, B, kappa, p_grid=p_grid, omega_grid=omega_grid, max_exp=700.0):
    """
    Numerically stable computation of alpha_{pω}, beta_{pω}.
    Prevents overflow by working in log-space.

    max_exp ~ 700 is safe for double precision.
    """

    # grids
    p = p_grid[:, None].astype(np.float64)
    w = omega_grid[None, :].astype(np.float64)

    # constants
    D = (1.0 / kappa) * np.log(A) - B

    # ----- common logarithmic pieces -----

    # log prefactor magnitude
    log_pref = -np.log(2 * np.pi) - np.log(p) - np.log(w)

    # log Gamma term
    lg = loggamma(1.0 + 1j * p / kappa)   # complex

    # omega^{- i p / kappa}
    phase_power = -(p / kappa) * np.log(w)

    # phase from D
    phase_D = -p * D

    # ----- alpha -----
    log_alpha_real = (
        log_pref
        + np.real(lg)
        + np.pi * p / (2.0 * kappa)
    )

    log_alpha_real = np.clip(log_alpha_real, -max_exp, max_exp)

    phase_alpha = (
        np.imag(lg)
        + phase_power
        + phase_D
        + w * B
        - np.pi / 2
    )

    alpha = np.exp(log_alpha_real) * np.exp(1j * phase_alpha)

    # ----- beta -----
    log_beta_real = (
        log_pref
        + np.real(lg)
        - np.pi * p / (2.0 * kappa)
    )

    log_beta_real = np.clip(log_beta_real, -max_exp, max_exp)

    phase_beta = (
        np.imag(lg)
        + phase_power
        + phase_D
        - w * B
        + np.pi / 2
    )

    beta = np.exp(log_beta_real) * np.exp(1j * phase_beta)

    return alpha.astype(np.complex128), beta.astype(np.complex128)

In [7]:
class MirrorDataset(Dataset):
    def __init__(self, n_samples):
        self.t = np.linspace(1000, 1500, N)
        self.samples = []

        for _ in range(n_samples):
            A, B, kappa = sample_parameters()
            z = mirror_trajectory(self.t, A, B, kappa)

            alpha, beta = compute_alpha_beta(A, B, kappa)

            self.samples.append((
                torch.tensor(z, dtype=torch.complex64),
                torch.tensor(alpha),
                torch.tensor(beta)
            ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        return self.samples[i]

In [8]:
class ComplexTanh(nn.Module):
    def forward(self, z):
        return torch.tanh(z.real) + 1j * torch.tanh(z.imag)

class ComplexMatrixLinear(nn.Module):
    def __init__(self, M):
        super().__init__()
        self.linear = nn.Linear(
            M * M, M * M, bias=True, dtype=torch.complex64
        )

    def forward(self, x):
        # x: (batch, M, M)
        b = x.shape[0]
        x = x.view(b, M * M)
        x = self.linear(x)
        return x.view(b, M, M)

In [9]:
class MatrixNet(nn.Module):
    def __init__(self, depth=2):
        super().__init__()

        self.embed = nn.Linear(N, M * M, dtype=torch.complex64)

        self.layers = nn.ModuleList(
            [ComplexMatrixLinear(M) for _ in range(depth)]
        )

        self.act = ComplexTanh()

    def forward(self, z):
        # z: (batch, N)
        x = self.embed(z).view(-1, M, M)

        for layer in self.layers:
            x = self.act(layer(x))

        return x

In [10]:
dataset = MirrorDataset(n_samples=100)
loader = DataLoader(dataset, batch_size=1, shuffle=True)

alpha_net = MatrixNet().to(device)
beta_net  = MatrixNet().to(device)

opt_a = torch.optim.Adam(alpha_net.parameters(), lr=LR)
opt_b = torch.optim.Adam(beta_net.parameters(), lr=LR)

def complex_mse(x, y):
    diff = x - y
    return (diff.real**2 + diff.imag**2).mean()

In [11]:
for epoch in range(EPOCHS):
    for z, alpha_t, beta_t in loader:
        z = z.to(device)
        alpha_t = alpha_t.to(device)
        beta_t = beta_t.to(device)

        opt_a.zero_grad()
        loss_a = complex_mse(alpha_net(z), alpha_t)
        loss_a.backward()
        opt_a.step()

        opt_b.zero_grad()
        loss_b = complex_mse(beta_net(z), beta_t)
        loss_b.backward()
        opt_b.step()

    if ((epoch+1) % 10 == 0):
        print(f"Epoch {epoch+1}: Loss for 'α': {loss_a.item():.6f} | Loss for 'β': {loss_b.item():.6f}")

Epoch 10: Loss for 'α': 0.009290 | Loss for 'β': 0.000026
Epoch 20: Loss for 'α': 0.007837 | Loss for 'β': 0.000022
Epoch 30: Loss for 'α': 0.010911 | Loss for 'β': 0.000042
Epoch 40: Loss for 'α': 0.010236 | Loss for 'β': 0.000042
Epoch 50: Loss for 'α': 0.006684 | Loss for 'β': 0.000053
Epoch 60: Loss for 'α': 0.010434 | Loss for 'β': 0.000057
Epoch 70: Loss for 'α': 0.009818 | Loss for 'β': 0.000068
Epoch 80: Loss for 'α': 0.004816 | Loss for 'β': 0.000063
Epoch 90: Loss for 'α': 0.009649 | Loss for 'β': 0.000106
Epoch 100: Loss for 'α': 0.009589 | Loss for 'β': 0.000092


## Checking for random values of A, B and $\kappa$

In [12]:
A, B, kappa = 0.6, 0.3, 0.25
t = np.linspace(1000, 1500, N)
z = mirror_trajectory(t, A, B, kappa)

alpha_true, beta_true = compute_alpha_beta(A, B, kappa)

with torch.no_grad():
    zt = torch.tensor(z, dtype=torch.complex64).unsqueeze(0).to(device)
    alpha_pred = alpha_net(zt)[0].cpu().numpy()
    beta_pred  = beta_net(zt)[0].cpu().numpy()

In [13]:
alpha_pred - alpha_true

array([[ 1.19+0.14j, -0.1 -0.53j, -0.22-0.02j, -0.06+0.18j,  0.03+0.17j,
         0.05+0.13j,  0.03+0.08j,  0.02+0.03j,  0.01+0.01j,  0.01+0.01j],
       [-0.58-0.19j,  0.07-0.22j,  0.  +0.17j,  0.07-0.01j, -0.1 -0.05j,
        -0.15+0.06j, -0.08+0.15j, -0.01+0.15j,  0.03+0.12j,  0.03+0.08j],
       [-0.64+0.45j,  0.12+0.21j, -0.12+0.18j,  0.04-0.06j, -0.11+0.04j,
         0.02+0.14j,  0.05+0.04j, -0.01+0.01j, -0.05+0.02j, -0.03+0.04j],
       [-0.44+0.13j, -0.12-0.12j, -0.08-0.25j,  0.02+0.04j, -0.12+0.04j,
         0.03+0.08j, -0.04-0.02j, -0.07+0.05j,  0.01+0.07j,  0.04+0.01j],
       [-0.15-0.35j, -0.13-0.j  ,  0.06+0.11j, -0.12+0.02j, -0.05-0.03j,
         0.  +0.08j, -0.03-0.05j, -0.07+0.04j,  0.01+0.01j, -0.02-0.04j],
       [ 0.28-0.09j, -0.04+0.15j,  0.16-0.11j,  0.04-0.08j,  0.01-0.02j,
        -0.07+0.03j, -0.  -0.04j, -0.04+0.02j,  0.  -0.01j, -0.05-0.02j],
       [-0.17+0.1j , -0.11+0.09j, -0.08-0.05j, -0.06+0.01j, -0.05+0.03j,
        -0.01-0.07j,  0.03-0.03j, -0.05-0.03j

In [14]:
np.abs(alpha_pred - alpha_true)

array([[1.2 , 0.53, 0.22, 0.19, 0.17, 0.14, 0.08, 0.04, 0.02, 0.01],
       [0.61, 0.23, 0.17, 0.07, 0.11, 0.16, 0.17, 0.15, 0.12, 0.08],
       [0.79, 0.24, 0.22, 0.07, 0.12, 0.14, 0.07, 0.01, 0.05, 0.05],
       [0.46, 0.17, 0.26, 0.05, 0.13, 0.08, 0.04, 0.09, 0.07, 0.04],
       [0.38, 0.13, 0.12, 0.12, 0.06, 0.08, 0.06, 0.08, 0.02, 0.04],
       [0.29, 0.15, 0.19, 0.09, 0.02, 0.08, 0.04, 0.05, 0.01, 0.06],
       [0.19, 0.14, 0.1 , 0.06, 0.05, 0.07, 0.04, 0.06, 0.05, 0.07],
       [0.26, 0.16, 0.09, 0.05, 0.07, 0.05, 0.03, 0.06, 0.04, 0.03],
       [0.38, 0.16, 0.1 , 0.08, 0.07, 0.03, 0.06, 0.05, 0.03, 0.03],
       [0.2 , 0.07, 0.12, 0.04, 0.08, 0.04, 0.03, 0.03, 0.04, 0.03]])

In [15]:
beta_pred - beta_true

array([[ 0.01+0.09j, -0.  -0.01j,  0.01-0.j  ,  0.  -0.01j, -0.01-0.j  ,
        -0.02+0.j  , -0.  +0.01j,  0.01+0.j  ,  0.01-0.j  ,  0.01-0.01j],
       [-0.  +0.01j,  0.01+0.j  , -0.  -0.01j, -0.  +0.01j, -0.  +0.01j,
         0.  +0.01j,  0.01-0.j  , -0.  -0.j  , -0.  +0.j  , -0.  +0.j  ],
       [ 0.  -0.j  ,  0.  +0.j  , -0.  +0.j  ,  0.  -0.j  , -0.  +0.j  ,
        -0.  -0.j  ,  0.  +0.j  ,  0.  +0.j  ,  0.  -0.j  , -0.  -0.j  ],
       [ 0.  +0.j  ,  0.  +0.j  , -0.  +0.j  , -0.01-0.01j,  0.01+0.01j,
         0.  -0.j  ,  0.  -0.j  , -0.  +0.j  ,  0.  -0.j  ,  0.  +0.01j],
       [-0.  -0.j  ,  0.  -0.j  ,  0.  +0.j  ,  0.  -0.j  ,  0.  -0.j  ,
        -0.  +0.j  ,  0.  +0.j  ,  0.  +0.j  , -0.  +0.j  , -0.  -0.j  ],
       [-0.  -0.01j, -0.  +0.j  ,  0.01-0.01j, -0.01-0.j  ,  0.  -0.j  ,
         0.01+0.j  , -0.  +0.j  , -0.  -0.j  ,  0.  +0.j  , -0.  -0.01j],
       [ 0.  +0.j  , -0.  +0.j  ,  0.  -0.j  ,  0.01-0.01j,  0.  -0.j  ,
        -0.  +0.j  ,  0.  +0.j  ,  0.  +0.j  

In [16]:
np.abs(beta_pred - beta_true)

array([[0.09, 0.01, 0.01, 0.01, 0.01, 0.02, 0.01, 0.01, 0.01, 0.01],
       [0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.01, 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.01, 0.01, 0.  , 0.  , 0.  , 0.  , 0.01],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.01, 0.  , 0.01, 0.01, 0.  , 0.01, 0.  , 0.  , 0.  , 0.01],
       [0.  , 0.  , 0.  , 0.02, 0.  , 0.  , 0.  , 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.  , 0.02, 0.  , 0.  ],
       [0.  , 0.  , 0.02, 0.01, 0.  , 0.02, 0.  , 0.  , 0.02, 0.  ],
       [0.  , 0.  , 0.01, 0.02, 0.  , 0.01, 0.  , 0.  , 0.02, 0.  ]])

## Checking for constraint: $\alpha^{\ast} \alpha^{T} - \beta \beta^{\dagger} = I$

In [17]:
alpha_pred.conj() @ alpha_pred.T - beta_pred @ beta_pred.conj().T

array([[ 0.27+0.j  ,  0.02-0.01j, -0.17+0.02j, -0.03-0.05j,  0.06-0.01j,
         0.  +0.01j,  0.05-0.j  , -0.01+0.02j, -0.  -0.06j, -0.  -0.03j],
       [ 0.02+0.01j,  0.07+0.j  ,  0.02-0.01j,  0.03+0.02j,  0.02+0.01j,
         0.02+0.02j, -0.02+0.01j, -0.01-0.j  , -0.  -0.01j, -0.01+0.01j],
       [-0.17-0.02j,  0.02+0.01j,  0.15+0.j  ,  0.04+0.04j, -0.03+0.02j,
         0.  +0.02j, -0.05+0.j  ,  0.01-0.02j, -0.01+0.05j, -0.01+0.01j],
       [-0.03+0.05j,  0.03-0.02j,  0.04-0.04j,  0.05+0.j  ,  0.  +0.02j,
         0.  +0.01j, -0.01+0.01j, -0.  +0.j  ,  0.  +0.01j,  0.01-0.j  ],
       [ 0.06+0.01j,  0.02-0.01j, -0.03-0.02j,  0.  -0.02j,  0.03-0.j  ,
         0.01+0.j  ,  0.01+0.01j, -0.01+0.j  ,  0.  -0.01j, -0.  -0.j  ],
       [ 0.  -0.01j,  0.02-0.02j,  0.  -0.02j,  0.  -0.01j,  0.01-0.j  ,
         0.03+0.j  , -0.  +0.01j, -0.  -0.j  ,  0.01+0.j  , -0.01+0.01j],
       [ 0.05+0.j  , -0.02-0.01j, -0.05-0.j  , -0.01-0.01j,  0.01-0.01j,
        -0.  -0.01j,  0.02+0.j  , -0.  +0.01j

In [18]:
np.abs(alpha_pred.conj() @ alpha_pred.T - beta_pred @ beta_pred.conj().T)

array([[0.27, 0.02, 0.17, 0.06, 0.06, 0.01, 0.05, 0.02, 0.06, 0.03],
       [0.02, 0.07, 0.02, 0.03, 0.02, 0.03, 0.02, 0.01, 0.01, 0.01],
       [0.17, 0.02, 0.15, 0.05, 0.04, 0.02, 0.05, 0.02, 0.05, 0.02],
       [0.06, 0.03, 0.05, 0.05, 0.02, 0.02, 0.02, 0.  , 0.01, 0.01],
       [0.06, 0.02, 0.04, 0.02, 0.03, 0.01, 0.02, 0.01, 0.01, 0.  ],
       [0.01, 0.03, 0.02, 0.02, 0.01, 0.03, 0.01, 0.01, 0.01, 0.01],
       [0.05, 0.02, 0.05, 0.02, 0.02, 0.01, 0.02, 0.01, 0.01, 0.01],
       [0.02, 0.01, 0.02, 0.  , 0.01, 0.01, 0.01, 0.01, 0.01, 0.01],
       [0.06, 0.01, 0.05, 0.01, 0.01, 0.01, 0.01, 0.01, 0.02, 0.01],
       [0.03, 0.01, 0.02, 0.01, 0.  , 0.01, 0.01, 0.01, 0.01, 0.01]],
      dtype=float32)

## Notes
- Range of t is (1000, 1500)
- Grid size is kept short (10) for now as the alpha and beta terms have exponents which blow up for higher values of p and omega
- Ranges of $A, B$ and $\kappa$ are also small for now
- Constraint equation is not satisfied as we're only taking 10x10 matrices.

### We might need a different implementation